<a href="https://colab.research.google.com/github/toeyldev/cmpe189-ai-photo-enhancement/blob/zahid-analysis/photo_enhancement_pipeline_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 0: Clone Toey's DnCNN branch
# contains: src/download_data.py, src/degrade_images.py,
#           src/dncnn_pytorch.py, src/dncnn_weights.py
#           requirements.txt

!git clone -b DnCNN https://github.com/toeyldev/cmpe189-ai-photo-enhancement.git
%cd cmpe189-ai-photo-enhancement
!pip install -r requirements.txt

Cloning into 'cmpe189-ai-photo-enhancement'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 122 (delta 44), reused 76 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 8.88 MiB | 11.90 MiB/s, done.
Resolving deltas: 100% (44/44), done.
/content/cmpe189-ai-photo-enhancement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.9 MB/s eta 0:00:00


In [ ]:
# Cell 1: Run Toey's pipeline scripts
# download_data.py  → downloads clean images from Flickr2K
# degrade_images.py → generates degraded versions

!python src/download_data.py
!python src/degrade_images.py

README.md: 100% 28.0/28.0 [00:00<00:00, 126kB/s]
Flickr2K.zip:  34% 3.98G/11.6G [00:34<01:47, 71.0MB/s]

**cv2: OpenCV (Open Source Computer Vision Library) in Python**

It's a library used for:
* image processing
* computer vision
* video processing

1. Read an image

img = cv2.imread("image.png")
img is a 3D numpy Array of numbers

- Loads image into a NumPy array
- img.shape → (height, width, channels)

2. Save an image

cv2.imwrite("output.png", img)

- Writes image to disk

3. Resize image

resized = cv2.resize(img, (256, 256))

or scale:

small = cv2.resize(img, None, fx=0.5, fy=0.5)

- Used to simulate low resolution

4. Convert color (VERY IMPORTANT)

rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

- OpenCV uses BGR, but matplotlib uses RGB

BGR = [Blue, Green, Red]

RGB = [Red, Green, Blue]

In [ ]:
# Cell 2: Generate degraded images
# Pipeline: clean → downsample → upsample → add noise → degraded

"""
clean image
   ↓
downsample → lose detail
   ↓
upsample → blur
   ↓
add noise → realistic degradation
   ↓
save → degraded image
"""

import cv2
import numpy as np
import os

os.makedirs("data/degraded", exist_ok=True)

#return: ["image_0.png", "image_1.png", ...]
#getFilename (Ex:image_0.png)
for fileName in os.listdir("data/clean"):

  #"data/clean" + "image_0.png"
  #builds: data/clean/image_0.png
  path = os.path.join("data/clean", fileName)

  # 1. clean image
  img = cv2.imread(path)
  #cv2.imread() takes a file path string and reads the image from disk into a NumPy array

  #if cv2.imread() fails: return None object
  if img is None:
        print(f"Could not read {path}")
        continue

  # 2. downsized image
  #We want: same size image, but lower quality
  #dsize (Width, Height) = None -> No hardcoded
  #fx=0.2 → new width = original width × 0.2 (20%)
  #fy=0.2 → new height = original height × 0.2 (20%)
  # INTER_AREA  → shrinking (averages pixels in the region, avoids blur)

  downSized = cv2.resize(img, None, fx=0.2, fy=0.2, interpolation=cv2.INTER_AREA)

  # 3. upsized image
  #Resize back to the original width and height
  #img.shape → (height, width, channels)
  # NumPy order: cv2.resize() → (width, height) # OpenCV order
  # INTER_CUBIC → enlarging (smooth, high-quality upscale, use 4x4 neighbour's pixel)
  upSized = cv2.resize(downSized, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_CUBIC)

  # 4. noisy image
  #noise = the disturbance
  #generate noise: mean = 0, std = 50 (controls noise strength), same shape as image
  #~68% of values → between -50 and +50
  noise = np.random.normal(0, 50, img.shape)

  """
  Original pixel: [100, 150, 200]
  Noise:          [+10,  -20,  +5]
  --------------------------------
  Result:         [110, 130, 205]
  """

  #noisy = the degraded image or noisy = upSizeImg + noise
  #0-255 = value of each pixel
  # astype(np.uint8): 123.7 → 123
  noisyImage = np.clip(upSized + noise, 0, 255).astype(np.uint8)

  savePath = os.path.join("data/degraded", fileName)
  cv2.imwrite(savePath, noisyImage)

print("Done creating degraded images.")
!ls data/degraded | head

In [ ]:
# Cell 3: Visualize Clean vs Degraded

import cv2
import matplotlib.pyplot as plt

clean_path    = cv2.imread("data/clean/image_0.png")
degraded_path = cv2.imread("data/degraded/image_0.png")

#OpenCV reads:      [B, G, R]  per pixel
#matplotlib wants:  [R, G, B]  per pixel
clean_rgb    = cv2.cvtColor(clean_path,    cv2.COLOR_BGR2RGB)   # ← missing line
degraded_rgb = cv2.cvtColor(degraded_path, cv2.COLOR_BGR2RGB)   # ← missing line

#plt.figure(): creates a new blank figure
# figsize=(10, 4):  width-height (in inches)
plt.figure(figsize=(10, 4)) # <- creates ONE shared figure

plt.subplot(1, 2, 1)       # <- draws inside that figure, position 1
plt.imshow(clean_rgb)
plt.title("Clean Image")
plt.axis("off")

plt.subplot(1, 2, 2)      # <- draws inside SAME figure, position 2
plt.imshow(degraded_rgb)
plt.title("Degraded Image")
plt.axis("off")

plt.show()

In [ ]:
#degraded image → DnCNN → denoised image

#Step 1: Install dependencies
!pip install torch torchvision

In [ ]:
# Cell 4: DnCNN Model Loading

import torch
import torch.nn as nn

# ── DEVICE SETUP ──────────────────────────────────────────────────────────────
# GPU (CUDA) = NVIDIA hardware + software that runs math in parallel → much faster
# Falls back to CPU if no GPU available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── STEP 1: DOWNLOAD PRETRAINED WEIGHTS ───────────────────────────────────────
# .pth = PyTorch weight file = dictionary of tensors {layer_name: learned_numbers}
# "color blind" = trained on RGB images (channels=3), handles any noise level
# These weights are the result of training on millions of images — we skip retraining
!wget -O dncnn_color_blind.pth https://github.com/cszn/KAIR/releases/download/v1.0/dncnn_color_blind.pth

# ── STEP 2: BUILD MODEL ARCHITECTURE ──────────────────────────────────────────
# Architecture must match the weight file exactly (20 layers, no BatchNorm)
# otherwise load_state_dict(strict=True) will throw an error
#
# Image flows through as a tensor: (C, H, W) = (3, height, width)
#
# Assembly line (20 layers):
#   First layer:   3 channels in  → 64 feature maps out   (RGB → patterns)
#   Middle layers: 64 → 64 (×18)                          (refine patterns)
#   Last layer:    64 feature maps → 3 channels out        (patterns → noise map)
#
# Each layer = Conv2d + ReLU
#   Conv2d → slides 3×3 window across image, detects patterns
#   ReLU   → kills negative values (adds complexity so model learns non-linear patterns)
#
# Key idea — residual learning:
#   The model does NOT output a clean image directly
#   It predicts the NOISE MAP, then subtracts it:
#   clean image = input - predicted_noise

class DnCNN(nn.Module):
    def __init__(self, channels=3, num_of_layers=20, features=64):
        super(DnCNN, self).__init__()

        layers = []

        # First layer: RGB (3ch) → 64 feature maps

        #padding=1:  add 1 pixel of zeros around the border -> cover the edge pixels fully

        #bias=True: a single extra number added after to shift the result up or down
        # without bias → output can only be 0 when input is 0
        # with bias    → output can be any value even when input is 0
        layers.append(nn.Conv2d(channels, features, 3, padding=1, bias=True))
        layers.append(nn.ReLU(inplace=True))

        # Middle layers: 64 → 64 (repeated 18 times, refining features)
        for _ in range(num_of_layers - 2):
            layers.append(nn.Conv2d(features, features, 3, padding=1, bias=True))
            layers.append(nn.ReLU(inplace=True))

        # Last layer: 64 feature maps → noise map (3ch, same size as input)
        layers.append(nn.Conv2d(features, channels, 3, padding=1, bias=True))

        # "model" name must match weight file keys: model.0, model.2, ... model.38
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        noise = self.model(x)   # run image through all 20 layers → get noise map
        return x - noise        # subtract noise → clean image (residual learning)

# ── STEP 3: LOAD WEIGHTS INTO MODEL ───────────────────────────────────────────
state_dict = torch.load("dncnn_color_blind.pth", map_location=device)


"""
# sometimes .pth is saved flat:
state_dict = {
    "model.0.weight": tensor,
    "model.0.bias":   tensor,
    ...
}

# sometimes .pth is saved nested under "params":
state_dict = {
    "params": {
        "model.0.weight": tensor,
        "model.0.bias":   tensor,
        ...
    }
}
"""
if "params" in state_dict:
    state_dict = state_dict["params"]  # unwrap if nested under "params" key
                                       # go one level deeper

model = DnCNN(channels=3, num_of_layers=20)
model.load_state_dict(state_dict, strict=True)  # strict=True: error if shapes don't match
model = model.to(device)   # move all weights to GPU memory
model.eval()               # inference mode: disables training-only behavior (faster, consistent)

print("RGB DnCNN model loaded successfully on", device)

## When do we use Pandas vs PyTorch?

---

### Pandas — Data Manipulation & Analysis
Think of it like **Excel in Python** — used for tables, CSVs, and structured data.
```python
import pandas as pd

df = pd.DataFrame(results)  # create table
df.to_csv("results.csv")    # save to CSV
df.mean()                   # compute averages
df.to_string()              # display table
```

**Use Pandas when you need to:**
- Store and display structured data (tables, spreadsheets)
- Compute statistics (mean, max, min)
- Read/write CSV files
- Organize and present results

---

### PyTorch — Deep Learning & Neural Networks
Think of it like the **engine that runs the AI model** — used for tensors, neural networks, and GPU computation.
```python
import torch

img_tensor = torch.from_numpy(img)  # convert image to tensor
model = DnCNN()                      # define neural network
output = model(img_tensor)           # run inference
torch.load("weights.pth")           # load pretrained weights
```

**Use PyTorch when you need to:**
- Define neural network architectures
- Run model inference
- Train models
- Work with tensors (multi-dimensional arrays optimized for math/GPU)

## Key Concepts from Task 3: Results Table Cell

---

### 1. PyTorch Tensor
Like a NumPy array but optimized for neural networks:
```
NumPy array    → general math/computation, CPU only
PyTorch tensor → same idea, but can run on GPU and tracks gradients for training
```

### 2. Why convert (H, W, C) → (C, H, W)?
```
NumPy   → (H, W, C)  channels LAST    → img[y][x][channel]
PyTorch → (C, H, W)  channels FIRST   → tensor[channel][y][x]
```

### 3. Why does unsqueeze(0) add a 1? (Batch Dimension)
```
(C, H, W)    → single image       ← model rejects this
(1, C, H, W) → batch of 1 image   ← model accepts this
(8, C, H, W) → batch of 8 images  ← also valid
```

### 4. Forward vs Backward Tensor Conversion
```
Forward (NumPy → Tensor):          Backward (Tensor → NumPy):
(H, W, C)                          (1, C, H, W)
  ↓ permute(2, 0, 1)                 ↓ squeeze()
(C, H, W)                          (C, H, W)
  ↓ unsqueeze(0)                     ↓ permute(1, 2, 0)
(1, C, H, W)                       (H, W, C)
  ↓ float()                          ↓ numpy()
tensor float32                     NumPy array float32
```

### 5. torch.no_grad()
```
without no_grad → tracks every operation (slow, uses extra memory)
with no_grad    → just computes the output  (fast, memory efficient)
```

### 6. PSNR vs SSIM
```
PSNR  → pixel-level accuracy  → higher is better (measured in dB)
SSIM  → structural similarity → higher is better (range 0.0 to 1.0)
```

In [ ]:
# Cell 5: Task 2 - PSNR/SSIM single image test
# quick test on ONE image to verify everything works before running the full loop

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import cv2
import torch
import numpy as np

# use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#compare_img: can be degraded image or denoised image
def compute_metrics(clean_img, compare_img):
    """
    Compute PSNR and SSIM between two images.
    Both images must be RGB uint8 NumPy arrays of the same shape.

    PSNR: higher is better (measures pixel-level accuracy)
    SSIM: higher is better, range 0-1 (measures structural similarity)
    """
    psnr_val = psnr(clean_img, compare_img)
    ssim_val = ssim(clean_img, compare_img, channel_axis=2)
    #channel_axis=2: tell SSIM which axis of NumPy contains the color channels
    #img.shape returns (height, width, channels)
    return psnr_val, ssim_val


# Load clean image (ground truth)
# data/ is inside cmpe189-ai-photo-enhancement/ which is our current working directory
clean = cv2.imread("data/clean/image_0.png")
clean_rgb = cv2.cvtColor(clean, cv2.COLOR_BGR2RGB) #[B, G, R] to [R, G, B]


# Load degraded image (model input / baseline)
degraded = cv2.imread("data/degraded/image_0.png")
degraded_rgb = cv2.cvtColor(degraded, cv2.COLOR_BGR2RGB)



# Compute baseline metrics (degraded vs clean)
psnr_degraded, ssim_degraded = compute_metrics(clean_rgb, degraded_rgb)

# run inference on image_0 to generate "out"
# make sure it is uint8 RGB at this point
img_norm   = degraded_rgb / 255.0                                               # min-max normalization to [0, 1]
img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float()  # NumPy (H,W,C) → tensor (1,C,H,W)
img_tensor = img_tensor.to(device)                                              # move tensor to GPU

with torch.no_grad():   # no_grad → fast, memory efficient (no gradient tracking needed at inference)
    output = model(img_tensor)

# convert output tensor back to uint8 NumPy image
# .cpu() moves tensor back from GPU to CPU before converting to NumPy
out = output.squeeze().cpu().permute(1, 2, 0).numpy()  # (1,C,H,W) → (H,W,C)
out = np.clip(out * 255, 0, 255).astype(np.uint8)      # [0,1] → [0,255], float32 → uint8

# Compute model metrics (denoised vs clean)
psnr_denoised, ssim_denoised = compute_metrics(clean_rgb, out)

print(f"PSNR  - Degraded: {psnr_degraded:.2f} dB  |  Denoised: {psnr_denoised:.2f} dB")
print(f"SSIM  - Degraded: {ssim_degraded:.4f}     |  Denoised: {ssim_denoised:.4f}")

In [ ]:
# Cell 6: Task 3 - Generate Results Table

import pandas as pd
import cv2
import os
import numpy as np
import torch

"""
Results Table Cell:
- Loops over all clean/degraded image pairs
- Runs DnCNN inference on each degraded image
- Computes PSNR and SSIM for:
    * degraded vs clean  (baseline)
    * denoised vs clean  (model output)
- Builds a pandas DataFrame and prints the results table
"""

#create a list (of dictionary later)
results = []

clean_dir    = "data/clean"
degraded_dir = "data/degraded"
enhanced_dir = "data/enhanced"
os.makedirs(enhanced_dir, exist_ok=True)

for fileName in sorted(os.listdir(clean_dir)):

    # --- Load images ---
    clean_path    = os.path.join(clean_dir, fileName)
    degraded_path = os.path.join(degraded_dir, fileName)

    clean    = cv2.imread(clean_path)
    degraded = cv2.imread(degraded_path)

    if clean is None or degraded is None:
        print(f"Skipping {fileName} - could not read file")
        continue

    # --- Convert BGR → RGB ---
    #cv2.cvtColor() always returns a 3D NumPy array
    clean_rgb    = cv2.cvtColor(clean, cv2.COLOR_BGR2RGB)
    degraded_rgb = cv2.cvtColor(degraded, cv2.COLOR_BGR2RGB)

    # --- Run DnCNN inference ---
    # normalize → tensor → model → back to uint8 RGB

    """
    min-max formula:  (x - min) / (max - min)
                = (x - 0)   / (255 - 0)
                = x / 255
    Since min is always 0 for images, the formula simplifies to just / 255.0.
    """

    img_norm   = degraded_rgb / 255.0                                               # min-max normalization to [0, 1]
    img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float()  # NumPy (H,W,C) → tensor (1,C,H,W)
    img_tensor = img_tensor.to(device)                                              # move tensor to GPU

    """
    torch.from_numpy(img_norm)   # NumPy array (H, W, C)  → PyTorch tensor (H, W, C)
    .permute(2, 0, 1)            # (H, W, C)              → (C, H, W)  PyTorch order
    .unsqueeze(0)                # (C, H, W)              → (1, C, H, W) adds batch dimension
    .float()                     # converts to float32 (model expects float, not uint8)
    """

    # with no_grad → PyTorch just computes the output (fast, memory efficient)
    # Gradients are only needed during training to update weights
    with torch.no_grad():
        output = model(img_tensor)  # model is Toey's DnCNN with pretrained weights loaded
                                    # output is a 4D PyTorch tensor containing the denoised image

    # Converting output tensor back to image
    """
    Forward (NumPy → Tensor):          Backward (Tensor → NumPy):
    (H, W, C)                          (1, C, H, W)
      ↓ permute(2, 0, 1)                 ↓ squeeze()
    (C, H, W)                          (C, H, W)
      ↓ unsqueeze(0)                     ↓ permute(1, 2, 0)
    (1, C, H, W)                       (H, W, C)
      ↓ float()                          ↓ numpy()
    tensor float32                     NumPy array float32
    """

    # .cpu() moves tensor back from GPU to CPU before converting to NumPy
    denoised = output.squeeze().cpu().permute(1, 2, 0).numpy()  # (1,C,H,W) → (H,W,C)

    # earlier we did: img_norm = degraded_rgb / 255.0  →  [0, 255] to [0.0, 1.0]
    # now we undo it: denoised * 255                   →  [0.0, 1.0] to [0.0, 255.0]
    denoised = np.clip(denoised * 255, 0, 255).astype(np.uint8)

    # --- Save enhanced image for Ryan ---
    save_path = os.path.join(enhanced_dir, fileName)
    # OpenCV saves images in BGR order, so convert RGB → BGR before saving
    denoised_bgr = cv2.cvtColor(denoised, cv2.COLOR_RGB2BGR)
    cv2.imwrite(save_path, denoised_bgr)

    # --- Compute metrics ---
    # This runs (PSNR & SSIM) on every image to build the full results table
    psnr_degraded, ssim_degraded = compute_metrics(clean_rgb, degraded_rgb)
    psnr_denoised, ssim_denoised = compute_metrics(clean_rgb, denoised)

    #results is a list of dictionaries: each loop iteration appends one dictionary (one row of data)
    #SSIM uses 4 decimal places because it's a small number between 0-1
    results.append({
        "image"         : fileName,
        "PSNR_degraded" : round(psnr_degraded, 2),
        "PSNR_denoised" : round(psnr_denoised, 2),
        "SSIM_degraded" : round(ssim_degraded, 4),
        "SSIM_denoised" : round(ssim_denoised, 4),
    })

# --- Build and display table ---
# Converts the list of dictionaries into a pandas table automatically
# each dictionary becomes one row, each key becomes one column
df = pd.DataFrame(results)

# Add average row at the bottom
# df.mean(numeric_only=True) computes the average of each numeric column
# skipping the "image" column (since you can't average strings)
# avg_row is a pandas Series
avg_row = df.mean(numeric_only=True).round(4)

# restoring the "image" label that df.mean() dropped because it couldn't average strings
avg_row["image"] = "AVERAGE"

"""
avg_row.to_frame()   # converts Series (1D) → DataFrame (2D column)
.T                   # transposes column → row (so it matches table shape)
pd.concat([df, ...]) # appends average row to bottom of table
ignore_index=True    # resets row numbering after concat
"""
df = pd.concat([df, avg_row.to_frame().T], ignore_index=True)

print(df.to_string(index=False))

# save to CSV for report
df.to_csv("results_table.csv", index=False)
print("Saved results to results_table.csv")

In [ ]:
# Cell 7: Zahid Khan - Comparison & Analysis Section
# computes per-image improvement from DnCNN denoising

#import pandas as pd

#df = pd.read_csv("results_table.csv")

# remove average row for per-image analysis
#df_images = df[df["image"] != "AVERAGE"].copy()

# compute improvement per image
# positive gain = model improved the image
# negative gain = model made it worse (should not happen with good weights)
#df_images["PSNR_gain"] = df_images["PSNR_denoised"] - df_images["PSNR_degraded"]
#df_images["SSIM_gain"] = df_images["SSIM_denoised"] - df_images["SSIM_degraded"]

# print before → after averages
#print("Average PSNR:", round(df_images["PSNR_degraded"].mean(), 2), "→", round(df_images["PSNR_denoised"].mean(), 2))
#print("Average SSIM:", round(df_images["SSIM_degraded"].mean(), 4), "→", round(df_images["SSIM_denoised"].mean(), 4))

# per-image improvement table
#print("\nImprovement check:")
#print(df_images[["image", "PSNR_gain", "SSIM_gain"]].to_string(index=False))

# Cell 7: Zahid Khan - Model Comparison & Analysis Section
# compares DnCNN vs SwinIR on the same 50 degraded images

import pandas as pd

# load both result tables
dncnn_df = pd.read_csv("results_table.csv")
swinir_df = pd.read_csv("swinir_results_table.csv")

# remove average rows
dncnn_images = dncnn_df[dncnn_df["image"] != "AVERAGE"].copy()
swinir_images = swinir_df[swinir_df["image"] != "AVERAGE"].copy()

# rename columns so they are easier to compare after merge
dncnn_images = dncnn_images.rename(columns={
    "PSNR_denoised": "PSNR_DnCNN",
    "SSIM_denoised": "SSIM_DnCNN"
})

swinir_images = swinir_images.rename(columns={
    "PSNR_denoised": "PSNR_SwinIR",
    "SSIM_denoised": "SSIM_SwinIR"
})

# keep only needed columns
dncnn_images = dncnn_images[["image", "PSNR_degraded", "SSIM_degraded", "PSNR_DnCNN", "SSIM_DnCNN"]]
swinir_images = swinir_images[["image", "PSNR_SwinIR", "SSIM_SwinIR"]]

# merge tables on image name
compare_df = pd.merge(dncnn_images, swinir_images, on="image")

# compute differences
compare_df["PSNR_diff"] = compare_df["PSNR_SwinIR"] - compare_df["PSNR_DnCNN"]
compare_df["SSIM_diff"] = compare_df["SSIM_SwinIR"] - compare_df["SSIM_DnCNN"]

# determine winner per image
compare_df["Winner"] = compare_df["PSNR_diff"].apply(lambda x: "SwinIR" if x > 0 else "DnCNN")

# summary values
avg_psnr_degraded = compare_df["PSNR_degraded"].mean()
avg_ssim_degraded = compare_df["SSIM_degraded"].mean()

avg_psnr_dncnn = compare_df["PSNR_DnCNN"].mean()
avg_ssim_dncnn = compare_df["SSIM_DnCNN"].mean()

avg_psnr_swinir = compare_df["PSNR_SwinIR"].mean()
avg_ssim_swinir = compare_df["SSIM_SwinIR"].mean()

swinir_wins = (compare_df["Winner"] == "SwinIR").sum()
dncnn_wins = (compare_df["Winner"] == "DnCNN").sum()

# print report-friendly summary
print("===== Zahid Model Comparison Summary =====")
print(f"Degraded  -> PSNR: {avg_psnr_degraded:.2f}, SSIM: {avg_ssim_degraded:.4f}")
print(f"DnCNN     -> PSNR: {avg_psnr_dncnn:.2f}, SSIM: {avg_ssim_dncnn:.4f}")
print(f"SwinIR    -> PSNR: {avg_psnr_swinir:.2f}, SSIM: {avg_ssim_swinir:.4f}")

print("\n===== Average Improvement Over DnCNN =====")
print(f"PSNR gain: {avg_psnr_swinir - avg_psnr_dncnn:.2f} dB")
print(f"SSIM gain: {avg_ssim_swinir - avg_ssim_dncnn:.4f}")

print("\n===== Winner Count =====")
print(f"SwinIR wins: {swinir_wins}/{len(compare_df)}")
print(f"DnCNN wins:  {dncnn_wins}/{len(compare_df)}")

print("\n===== First 10 Image Comparisons =====")
print(compare_df[["image", "PSNR_DnCNN", "PSNR_SwinIR", "PSNR_diff", "Winner"]].head(10).to_string(index=False))

# save comparison table for report use
compare_df.to_csv("dncnn_vs_swinir_comparison.csv", index=False)
print("\nSaved comparison table to dncnn_vs_swinir_comparison.csv")

In [ ]:
# Cell 8: Ryan Darghous - Visualization Section
# side-by-side comparison: clean vs degraded vs enhanced (DnCNN output)

import cv2
import matplotlib.pyplot as plt

name = "image_0.png"

clean    = cv2.cvtColor(cv2.imread(f"data/clean/{name}"),    cv2.COLOR_BGR2RGB)
degraded = cv2.cvtColor(cv2.imread(f"data/degraded/{name}"), cv2.COLOR_BGR2RGB)
enhanced = cv2.cvtColor(cv2.imread(f"data/enhanced/{name}"), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(clean)
plt.title("Clean (Ground Truth)")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(degraded)
plt.title("Degraded (Model Input)")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(enhanced)
plt.title("Enhanced (DnCNN Output)")
plt.axis("off")

plt.suptitle("DnCNN Image Enhancement Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("comparison_figure.png", dpi=150, bbox_inches="tight")
plt.show()

print("Figure saved to comparison_figure.png")

---
## Check-in 4
### SwinIR Model: Second Model for Comparison vs DnCNN

**SwinIR** (Swin Transformer for Image Restoration) is a transformer-based model — newer and more powerful than DnCNN.
We use the color denoising version trained at noise level σ=50, which matches our degradation pipeline exactly.

**Why SwinIR instead of ESRGAN?**
- ESRGAN is designed for super-resolution (upscaling) — our pipeline already upscales back to original size
- SwinIR handles both blur AND noise — matches our degradation exactly
- SwinIR generally outperforms DnCNN on complex images

**Same pipeline as DnCNN — just a different model:**

In [ ]:
# ── Check-in 4: SwinIR Model Setup (Thao) ────────────────────────────────────
# SwinIR uses Swin Transformer architecture (like GPT but for images)
# We need to clone the SwinIR repo to get the model architecture

import sys
import torch

# Step 1: Clone SwinIR repo (contains model architecture definition)
!git clone https://github.com/JingyunLiang/SwinIR.git

# add SwinIR to Python path so we can import from it
sys.path.insert(0, 'SwinIR')

# Step 2: Download pretrained weights
# noise50 = trained specifically for Gaussian noise sigma=50
# this matches our degradation pipeline (we also use sigma=50)
!wget -O swinir_color_dn50.pth https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/005_colorDN_DFWB_s128w8_SwinIR-M_noise50.pth

# Step 3: Load SwinIR model
from models.network_swinir import SwinIR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# build model architecture
# these parameters must match the pretrained weight file exactly
swinir_model = SwinIR(
    upscale=1,           # 1 = no upscaling (denoising only, not super-resolution)
    in_chans=3,          # 3 input channels (RGB)
    img_size=128,        # training patch size
    window_size=8,       # transformer window size
    img_range=1.0,       # pixel range [0, 1]
    depths=[6,6,6,6,6,6],       # depth of each transformer stage
    embed_dim=180,               # embedding dimension
    num_heads=[6,6,6,6,6,6],    # attention heads per stage
    mlp_ratio=2,                 # MLP expansion ratio
    upsampler='',                # no upsampler (denoising only)
    resi_connection='1conv'      # residual connection type
)

# load pretrained weights
pretrained = torch.load('swinir_color_dn50.pth', map_location=device)

# unwrap 'params' key if present (same pattern as DnCNN)
if 'params' in pretrained:
    pretrained = pretrained['params']

swinir_model.load_state_dict(pretrained, strict=True)
swinir_model = swinir_model.to(device)
swinir_model.eval()

print("SwinIR model loaded successfully on", device)

 ## ⚠️ Problem: GPU Out of Memory on Large Images

When running SwinIR on the full Flickr2K images (2040 × 1356 pixels), the GPU ran out of memory:


**Why it happened:**
SwinIR is a large transformer model. Processing a full 2040×1356 image at once requires ~5.5 GB of GPU memory — more than what was available after the model itself was loaded.

---

##  Fix: Tile-Based Inference

Instead of processing the full image at once, we split it into small **256×256 tiles**, run SwinIR on each tile separately, then stitch the results back together.

**Why tiles work:**
- Each 256×256 tile only needs ~0.5 GB GPU → fits easily
- Tiles overlap by 32 pixels at the edges to prevent visible seam lines
- Overlapping regions are averaged together for smooth transitions
- torch.cuda.empty_cache() is called after each image to free unused GPU memory

**Real world analogy:**
Painting a huge wall — you can't reach the whole wall at once, so you divide it into small sections, paint one section at a time, and step back to see one complete painting.


In [ ]:
# ── Check-in 4: SwinIR Inference on all 50 images (Thao) ─────────────────────
# Uses tile-based inference to avoid GPU out of memory on large images

"""
1. Added tile_inference() function
   → splits image into 256x256 overlapping tiles
   → runs model on each tile separately
   → stitches results back together
   → each tile uses only ~0.5GB GPU instead of 5.5GB

2. Added torch.cuda.empty_cache() after each image
   → releases unused GPU memory between images
   → prevents memory from accumulating

"""


import cv2
import numpy as np
import os
import pandas as pd
import torch
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# directories
clean_dir    = "data/clean"
degraded_dir = "data/degraded"
swinir_dir   = "data/swinir_enhanced"
os.makedirs(swinir_dir, exist_ok=True)

window_size = 8
tile_size   = 256   # process image in 256x256 tiles → fits in GPU memory
tile_overlap = 32   # overlap between tiles to avoid seam artifacts

def tile_inference(model, img_tensor, tile_size, tile_overlap, window_size, device):
    """
    Split large image into overlapping tiles, run model on each tile,
    then stitch tiles back together.
    Avoids GPU out of memory on large images.
    """
    b, c, h, w = img_tensor.shape
    output = torch.zeros_like(img_tensor)
    count  = torch.zeros_like(img_tensor)

    stride = tile_size - tile_overlap

    for y in range(0, h, stride):
        for x in range(0, w, stride):
            # tile boundaries
            y_end = min(y + tile_size, h)
            x_end = min(x + tile_size, w)
            y_start = y_end - tile_size if y_end - y < tile_size else y
            x_start = x_end - tile_size if x_end - x < tile_size else x

            # extract tile
            tile = img_tensor[:, :, y_start:y_end, x_start:x_end]

            # pad tile so dims divisible by window_size
            _, _, th, tw = tile.shape
            th_pad = (window_size - th % window_size) % window_size
            tw_pad = (window_size - tw % window_size) % window_size
            if th_pad > 0 or tw_pad > 0:
                tile = torch.nn.functional.pad(tile, (0, tw_pad, 0, th_pad), mode='reflect')

            # run model on tile
            with torch.no_grad():
                tile_out = model(tile)

            # crop back to original tile size
            tile_out = tile_out[:, :, :th, :tw]

            # accumulate output
            output[:, :, y_start:y_end, x_start:x_end] += tile_out
            count[:, :, y_start:y_end, x_start:x_end]  += 1

    # average overlapping regions
    output = output / count
    return output

swinir_results = []

for fileName in sorted(os.listdir(clean_dir)):

    # --- Load images ---
    clean    = cv2.imread(os.path.join(clean_dir,    fileName))
    degraded = cv2.imread(os.path.join(degraded_dir, fileName))

    if clean is None or degraded is None:
        print(f"Skipping {fileName} - could not read file")
        continue

    # --- Convert BGR → RGB ---
    clean_rgb    = cv2.cvtColor(clean,    cv2.COLOR_BGR2RGB)
    degraded_rgb = cv2.cvtColor(degraded, cv2.COLOR_BGR2RGB)

    # --- Normalize to [0, 1] ---
    img_norm = degraded_rgb / 255.0

    # --- NumPy → PyTorch tensor → GPU ---
    # (H, W, C) → (1, C, H, W)
    img_tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).float().to(device)

    # --- Run SwinIR inference in tiles ---
    output = tile_inference(swinir_model, img_tensor, tile_size, tile_overlap, window_size, device)

    # --- Convert output back to NumPy image ---
    # (1, C, H, W) → (H, W, C)
    denoised = output.squeeze().cpu().permute(1, 2, 0).numpy()

    # --- Undo normalization: [0, 1] → [0, 255] ---
    denoised = np.clip(denoised * 255, 0, 255).astype(np.uint8)

    # --- Save enhanced image ---
    save_path = os.path.join(swinir_dir, fileName)
    denoised_bgr = cv2.cvtColor(denoised, cv2.COLOR_RGB2BGR)
    cv2.imwrite(save_path, denoised_bgr)

    # --- Compute metrics ---
    psnr_degraded = psnr(clean_rgb, degraded_rgb)
    ssim_degraded = ssim(clean_rgb, degraded_rgb, channel_axis=2)
    psnr_swinir   = psnr(clean_rgb, denoised)
    ssim_swinir   = ssim(clean_rgb, denoised, channel_axis=2)

    swinir_results.append({
        "image"         : fileName,
        "PSNR_degraded" : round(psnr_degraded, 2),
        "PSNR_swinir"   : round(psnr_swinir,   2),
        "SSIM_degraded" : round(ssim_degraded, 4),
        "SSIM_swinir"   : round(ssim_swinir,   4),
    })

    print(f"{fileName}  PSNR: {psnr_degraded:.2f} → {psnr_swinir:.2f}  SSIM: {ssim_degraded:.4f} → {ssim_swinir:.4f}")

    # free GPU memory after each image
    torch.cuda.empty_cache()

# --- Build results table ---
df_swinir = pd.DataFrame(swinir_results)
avg_row = df_swinir.mean(numeric_only=True).round(4)
avg_row["image"] = "AVERAGE"
df_swinir = pd.concat([df_swinir, avg_row.to_frame().T], ignore_index=True)

print("\n" + df_swinir.to_string(index=False))

df_swinir.to_csv("swinir_results_table.csv", index=False)
print("\nSaved to swinir_results_table.csv")

In [ ]:
# ── Check-in 4: DnCNN vs SwinIR Comparison Table (Thao) ──────────────────────

import pandas as pd

# load both results
df_dncnn  = pd.read_csv("results_table.csv")
df_swinir = pd.read_csv("swinir_results_table.csv")

# remove average rows for merge
df_dncnn_imgs  = df_dncnn[df_dncnn["image"]   != "AVERAGE"].copy()
df_swinir_imgs = df_swinir[df_swinir["image"]  != "AVERAGE"].copy()

# merge on image name
df_compare = df_dncnn_imgs[["image", "PSNR_denoised", "SSIM_denoised"]].merge(
    df_swinir_imgs[["image", "PSNR_swinir", "SSIM_swinir"]],
    on="image"
)

# rename for clarity
df_compare = df_compare.rename(columns={
    "PSNR_denoised": "PSNR_DnCNN",
    "SSIM_denoised": "SSIM_DnCNN",
})

# which model won per image?
df_compare["PSNR_diff"] = (df_compare["PSNR_swinir"] - df_compare["PSNR_DnCNN"]).round(2)
df_compare["winner"]    = df_compare["PSNR_diff"].apply(lambda x: "SwinIR" if x > 0 else "DnCNN")

# add average row
avg = df_compare[["PSNR_DnCNN", "PSNR_swinir", "SSIM_DnCNN", "SSIM_swinir", "PSNR_diff"]].mean().round(4)
avg["image"]  = "AVERAGE"
avg["winner"] = "SwinIR" if avg["PSNR_diff"] > 0 else "DnCNN"
df_compare = pd.concat([df_compare, avg.to_frame().T], ignore_index=True)

print(df_compare.to_string(index=False))

# summary
avg_dncnn  = float(df_dncnn[df_dncnn["image"]   == "AVERAGE"]["PSNR_denoised"].values[0])
avg_swinir = float(df_swinir[df_swinir["image"]  == "AVERAGE"]["PSNR_swinir"].values[0])

print(f"\nAverage PSNR — DnCNN: {avg_dncnn} dB  |  SwinIR: {avg_swinir} dB")
print(f"Winner: {'SwinIR' if avg_swinir > avg_dncnn else 'DnCNN'} by {abs(avg_swinir - avg_dncnn):.2f} dB")

In [ ]:
# ── Check-in 4: 4-Panel Visualization (Thao) ──────────────────────────────────
# Clean | Degraded | DnCNN | SwinIR

import cv2
import matplotlib.pyplot as plt

name = "image_0.png"

clean    = cv2.cvtColor(cv2.imread(f"data/clean/{name}"),           cv2.COLOR_BGR2RGB)
degraded = cv2.cvtColor(cv2.imread(f"data/degraded/{name}"),        cv2.COLOR_BGR2RGB)
dncnn    = cv2.cvtColor(cv2.imread(f"data/enhanced/{name}"),        cv2.COLOR_BGR2RGB)
swinir   = cv2.cvtColor(cv2.imread(f"data/swinir_enhanced/{name}"), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(clean);    axes[0].set_title("Clean (Ground Truth)");  axes[0].axis("off")
axes[1].imshow(degraded); axes[1].set_title("Degraded (Model Input)"); axes[1].axis("off")
axes[2].imshow(dncnn);    axes[2].set_title("DnCNN Output");           axes[2].axis("off")
axes[3].imshow(swinir);   axes[3].set_title("SwinIR Output");          axes[3].axis("off")

plt.suptitle("Model Comparison: DnCNN vs SwinIR", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("comparison_figure_checkin4.png", dpi=150, bbox_inches="tight")
plt.show()

print("Figure saved to comparison_figure_checkin4.png")